# Sentiment Analysis Using RNN and LSTM
### A Comparative Study of Long-Term Dependency Learning in Text Sequences

**Author:** Deep Learning Mini Project Submission  
**Dataset:** IMDB Large Movie Review Dataset (50,000 samples)  
**Objective:** Build, train, evaluate, and compare Simple Recurrent Neural Networks (Simple RNN) and Long Short-Term Memory (LSTM) models under identical experimental conditions, specifically analyzing how their memory retention and gradient propagation dynamics differ across text sequence distances.

## 1. Environment Setup and Imports

In [ ]:
import os
import re
import sys
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, f1_score

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")

## 2. Dataset Loading: IMDB Movie Reviews

In [ ]:
from tensorflow.keras.datasets import imdb

VOCAB_SIZE = 10000
MAX_LEN = 150
EMBEDDING_DIM = 64
HIDDEN_UNITS = 64

(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = imdb.load_data(num_words=VOCAB_SIZE)
print(f"Train samples: {len(X_train_raw)}, Test samples: {len(X_test_raw)}")

# Decode sample review to verify data structure
word_index = imdb.get_word_index()
inv_word_index = {i + 3: w for w, i in word_index.items()}
inv_word_index[0] = "<PAD>"
inv_word_index[1] = "<START>"
inv_word_index[2] = "<UNK>"
inv_word_index[3] = "<UNUSED>"

sample_decoded = " ".join([inv_word_index.get(i, "?") for i in X_train_raw[0][:30]])
print("Sample Decoded (First 30 words):", sample_decoded)

## 3. NLP Preprocessing: Sequence Padding & Train/Val Split
Both architectures receive identical padded numerical representations.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_raw, maxlen=MAX_LEN, padding='post', truncating='post')

# Validation split (20,000 train, 5,000 validation)
X_train = X_train_pad[:20000]
y_train = y_train_raw[:20000].astype('float32')

X_val = X_train_pad[20000:25000]
y_val = y_train_raw[20000:25000].astype('float32')

X_test = X_test_pad
y_test = y_test_raw.astype('float32')

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

## 4. Simple RNN Architecture
Standard Elman recurrence with hidden state updates: $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)$.

In [ ]:
def build_simple_rnn():
    model = models.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
        layers.SimpleRNN(units=HIDDEN_UNITS, activation='tanh'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name='Simple_RNN_Model')
    
    model.compile(optimizer=optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

rnn_model = build_simple_rnn()
rnn_model.summary()

## 5. LSTM Architecture
Gated recurrent neural network featuring Forget, Input, Output gates, and Constant Error Carousel (CEC) Cell State.

In [ ]:
def build_lstm():
    model = models.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
        layers.LSTM(units=HIDDEN_UNITS),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name='LSTM_Model')
    
    model.compile(optimizer=optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm()
lstm_model.summary()

## 6. Model Training & Validation Curves

In [ ]:
EPOCHS = 4
BATCH_SIZE = 128

print("Training Simple RNN...")
t0 = time.time()
rnn_history = rnn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
rnn_train_time = time.time() - t0

print("\nTraining LSTM...")
t0 = time.time()
lstm_history = lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
lstm_train_time = time.time() - t0

print(f"\nTraining Times -> Simple RNN: {rnn_train_time:.1f}s | LSTM: {lstm_train_time:.1f}s")

## 7. Comparative Evaluation on Held-Out Test Set (25,000 samples)

In [ ]:
y_probs_rnn = rnn_model.predict(X_test, batch_size=128).flatten()
y_pred_rnn = (y_probs_rnn >= 0.5).astype(int)

y_probs_lstm = lstm_model.predict(X_test, batch_size=128).flatten()
y_pred_lstm = (y_probs_lstm >= 0.5).astype(int)

print("--- Simple RNN Classification Report ---")
print(classification_report(y_test, y_pred_rnn))

print("--- LSTM Classification Report ---")
print(classification_report(y_test, y_pred_lstm))

## 8. Confusion Matrix & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrices
cm_rnn = confusion_matrix(y_test, y_pred_rnn)
cm_lstm = confusion_matrix(y_test, y_pred_lstm)

sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Simple RNN Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Purples', ax=axes[1])
axes[1].set_title("LSTM Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 9. Long-Term Dependency Diagnostic Experiment
Testing whether Simple RNN forgets earlier tokens when distance between negation and sentiment word increases.

In [ ]:
# Controlled synthetic probes
test_cases = [
    "This was not a good film.",
    "This was not a movie with multiple characters good film.",
    "This was not a movie featuring standard plot elements, typical dialogue, normal pacing, ordinary editing, and predictable sequences good film."
]

print("Controlled Long-Distance Negation Probes:")
for sent in test_cases:
    tokens = [word_index.get(w, 2) + 3 for w in sent.lower().split()]
    padded = pad_sequences([tokens], maxlen=MAX_LEN, padding='post')
    p_rnn = rnn_model.predict(padded, verbose=0)[0][0]
    p_lstm = lstm_model.predict(padded, verbose=0)[0][0]
    print(f"Sentence: '{sent}'")
    print(f" -> RNN Prob: {p_rnn:.4f} ({'Pos' if p_rnn>=0.5 else 'Neg'}) | LSTM Prob: {p_lstm:.4f} ({'Pos' if p_lstm>=0.5 else 'Neg'})\n")

## 10. Conclusion & Academic Insights
1. **Vanishing Gradient:** Simple RNN gradients diminish exponentially due to repeated multiplication of $W_{hh}^T$. In long sentences, early tokens fail to propagate error gradients.
2. **LSTM Constant Error Carousel (CEC):** The additive cell state updates allow gradients to flow backward uninhibited, correctly identifying long-distance negations.
3. **Computational Trade-Off:** Simple RNN requires ~4x fewer floating point operations per recurrent step, making it much faster to train when sequences are short.